# V3 HMM Model Selection and Parameter Estimation
### Master Thesis: Revisiting Delegation Theory in the Age of AI: Dynamic Algorithm Appreciation and Aversion in Triadic Organizational Relationships

This notebook estimates the V3 two-emission hidden Markov model for triadic delegation behavior. It compares single-period and three-period transition specifications across `J=2`, `J=3`, and `J=4` latent states, requires strict optimizer convergence for every candidate, and exports the final model-selection results.

Final selected model after V3 emission-control calibration: `single_period, J=3`.

---

## Table of Contents

1. [V3 Data and Variable Construction](#1-v3-data-and-variable-construction)
   - 1.1 [Imports, Helpers, and Variable Lists](#11-imports-helpers-and-variable-lists)
   - 1.2 [V3 Variable Specification](#12-v3-variable-specification)
   - 1.3 [Sequence Loading and Scaling](#13-sequence-loading-and-scaling)
2. [Hidden Markov Model Estimation](#2-hidden-markov-model-estimation)
   - 2.1 [Model Structure and Inference Algorithms](#21-model-structure-and-inference-algorithms)
   - 2.2 [Batched Maximum Likelihood Estimator](#22-batched-maximum-likelihood-estimator)
   - 2.3 [Converged V3 Model Selection](#23-converged-v3-model-selection)
3. [Final Model Diagnostics and Artifact Export](#3-final-model-diagnostics-and-artifact-export)
   - 3.1 [Selected Model Data and Dimensions](#31-selected-model-data-and-dimensions)
   - 3.2 [Final Selected Fit Diagnostics](#32-final-selected-fit-diagnostics)
   - 3.3 [Artifact Export](#33-artifact-export)


---
<a id="1-v3-data-and-variable-construction"></a>
## 1. V3 Data and Variable Construction

<a id="11-imports-helpers-and-variable-lists"></a>
### 1.1 Imports, Helpers, and Variable Lists


In [ ]:
# ============================================================
# 1. Imports & Helpers
# ============================================================
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from multiprocessing.pool import ThreadPool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

# ---- resolve V3 data path ----
MANUAL_XLSX_PATH = None
PREFERRED_DATASETS = [
    Path("../Datasets/Triadic_Delegation_Analysis_Dataset_v3_state_trend_calibrated.xlsx"),
    Path("Datasets/Triadic_Delegation_Analysis_Dataset_v3_state_trend_calibrated.xlsx"),
    Path("../Datasets/Triadic_Delegation_Analysis_Dataset_v3.xlsx"),
    Path("Datasets/Triadic_Delegation_Analysis_Dataset_v3.xlsx"),
    Path("Triadic_Delegation_Analysis_Dataset_v3.xlsx"),
]
DATA_PATH = None
if MANUAL_XLSX_PATH:
    DATA_PATH = Path(MANUAL_XLSX_PATH)
else:
    for p in PREFERRED_DATASETS:
        if p.exists():
            DATA_PATH = p
            break
assert DATA_PATH is not None and DATA_PATH.exists(),     f"Data file not found. Tried: {PREFERRED_DATASETS}"
print(f"Data file: {DATA_PATH.resolve()}")


EMISSION_COLS = [
    "ai_authority_share",
    "escalation_share",
]

# Full emission-side control set retained for the calibrated V3 synthetic DGP.
CONTROL_COLS = [
    "decision_latency",
    "demand_volatility",
    "forecast_accuracy",
    "performance_pressure",
    "recent_negative_shock",
    "supply_disruptions",
    "target_difficulty",
    "task_complexity",
]

SINGLE_PERIOD_TRANSITION_COLS = [
    "team_t_minus_1_vs_team_t",
    "team_vs_peer_average",
    "target_attainment",
]

THREE_PERIOD_TRANSITION_COLS = [
    "team_prev3_avg_vs_team_t",
    "team_vs_peer_average_3period",
    "target_attainment_3period",
]

TRANSITION_COLS = list(SINGLE_PERIOD_TRANSITION_COLS)
TRANSITION_SPECS = {
    "single_period": list(SINGLE_PERIOD_TRANSITION_COLS),
    "three_period": list(THREE_PERIOD_TRANSITION_COLS),
}


VARIABLE_LABELS = {
    "ai_authority_share": "AI Authority Share",
    "escalation_share": "Escalation Share",
    "decision_latency": "Decision Latency",
    "demand_volatility": "Demand Volatility",
    "forecast_accuracy": "Forecast Accuracy",
    "performance_pressure": "Performance Pressure",
    "recent_negative_shock": "Recent Negative Shock",
    "supply_disruptions": "Supply Disruptions",
    "target_difficulty": "Target Difficulty",
    "task_complexity": "Task Complexity",
    "team_t_minus_1_vs_team_t": "Team(t-1) vs. Team(t)",
    "team_vs_peer_average": "Team vs. Peer Average",
    "target_attainment": "Target Attainment",
    "team_prev3_avg_vs_team_t": "Team previous 3-cycle average vs. Team(t)",
    "team_vs_peer_average_3period": "Team vs. Peer Average, trailing 3-cycle mean",
    "target_attainment_3period": "Target Attainment, trailing 3-cycle rate",
}

THREE_PERIOD_DEFINITION = (
    "team_prev3_avg_vs_team_t = current composite KPI minus the manager's "
    "mean composite KPI over the previous three cycles; "
    "team_vs_peer_average_3period and target_attainment_3period are trailing "
    "three-cycle manager-level rolling means."
)


def softmax(z, axis=-1):
    """Numerically stable softmax (works on 1-D vectors and row-wise on 2-D)."""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    """Numerically stable log-softmax."""
    return z - logsumexp(z, axis=axis, keepdims=True)


def log_gaussian_diag(y, mean, log_sigma):
    """Scalar version (kept for reference)."""
    sigma2 = np.exp(2 * log_sigma)
    return -0.5 * (
        np.sum(np.log(2 * np.pi * sigma2))
        + np.sum((y - mean) ** 2 / sigma2)
    )

print("Section 1 - Imports, helpers, and V3 variable lists loaded.")


<a id="12-v3-variable-specification"></a>
### 1.2 V3 Variable Specification

The V3 model uses the generated `panel_manager_period` sheet directly. Transparency variables are excluded from the analysis dataset.

Emissions:
- `ai_authority_share`
- `escalation_share`

Emission-side control variables:
- `decision_latency`
- `demand_volatility`
- `forecast_accuracy`
- `performance_pressure`
- `recent_negative_shock`
- `supply_disruptions`
- `target_difficulty`
- `task_complexity`

Transition specifications compared in model selection:
- `single_period`: `team_t_minus_1_vs_team_t`, `team_vs_peer_average`, `target_attainment`
- `three_period`: `team_prev3_avg_vs_team_t`, `team_vs_peer_average_3period`, `target_attainment_3period`

The three-period variables are derived from manager-level trailing three-cycle history to test whether smoother recent performance history fits better than single-period movement.


In [ ]:
# ============================================================
# 2. Validate and Build V3 Transition Variables
# ============================================================

def build_benchmarks(df):
    """Validate V3 variables and derive the 3-period transition specification."""
    df = df.sort_values(["manager_id", "period_id"]).copy()

    required_cols = [
        "manager_id",
        "period_id",
        "composite_kpi_score",
        *EMISSION_COLS,
        *CONTROL_COLS,
        *SINGLE_PERIOD_TRANSITION_COLS,
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required V3 columns: {missing}")

    numeric_cols = [
        "period_id",
        "composite_kpi_score",
        *EMISSION_COLS,
        *CONTROL_COLS,
        *SINGLE_PERIOD_TRANSITION_COLS,
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    g = df.groupby("manager_id", group_keys=False)

    prev3_kpi = g["composite_kpi_score"].transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
    df["team_prev3_avg_vs_team_t"] = (
        df["composite_kpi_score"] - prev3_kpi
    ).fillna(0.0)

    df["team_vs_peer_average_3period"] = g["team_vs_peer_average"].transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    df["target_attainment_3period"] = g["target_attainment"].transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )

    for col in THREE_PERIOD_TRANSITION_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

print("Section 2 - V3 variables validated; three-period transition variables derived.")
print("Three-period definition:", THREE_PERIOD_DEFINITION)


<a id="13-sequence-loading-and-scaling"></a>
### 1.3 Sequence Loading and Scaling

Load the V3 `panel_manager_period` sheet, validate required variables, scale emissions/covariates/controls, and form balanced per-manager sequences.


In [ ]:
# ============================================================
# 3. Data Loader
# ============================================================

@dataclass
class HMMData:
    Y: List[np.ndarray]      # emission sequences
    X: List[np.ndarray]      # transition covariates
    Z: List[np.ndarray]      # emission controls
    ids: List[str]
    periods: List[np.ndarray]
    y_scaler: StandardScaler
    x_scaler: StandardScaler
    z_scaler: StandardScaler


def load_sequences(xlsx_path, transition_cols_override: Optional[List[str]] = None):
    # Cache raw sheet to avoid repeated Excel I/O in model selection.
    panel_cache_key = "_panel_manager_period_cache_v3"

    if panel_cache_key not in globals():
        globals()[panel_cache_key] = pd.read_excel(xlsx_path, sheet_name="panel_manager_period")

    df = globals()[panel_cache_key].copy()

    # Safety: analysis file should not contain latent truth columns.
    forbidden = ["latent_state_true", "latent_state_true_next"]
    if any(c in df.columns for c in forbidden):
        df = df.drop(columns=[c for c in forbidden if c in df.columns])
        print("  Dropped latent truth columns to proceed with analysis data.")

    df = build_benchmarks(df)

    emission_cols = list(EMISSION_COLS)
    transition_cols = list(transition_cols_override) if transition_cols_override is not None else list(TRANSITION_COLS)
    control_cols = list(CONTROL_COLS)

    globals()["emission_cols"] = emission_cols
    globals()["transition_cols"] = transition_cols
    globals()["control_cols"] = control_cols

    missing = [c for c in emission_cols + transition_cols + control_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required model columns: {missing}")

    df = df.dropna(subset=emission_cols + transition_cols + control_cols)

    Y_list, X_list, Z_list = [], [], []
    ids, periods = [], []

    for mid, g in df.groupby("manager_id"):
        g = g.sort_values("period_id")
        Y = g[emission_cols].to_numpy(float)
        X = g[transition_cols].to_numpy(float)
        Z = g[control_cols].to_numpy(float) if control_cols else np.zeros((len(g), 0), dtype=float)
        if len(Y) < 3:
            continue
        Y_list.append(Y)
        X_list.append(X)
        Z_list.append(Z)
        ids.append(mid)
        periods.append(g["period_id"].to_numpy())

    if not Y_list:
        raise ValueError("No valid manager sequences after filtering missing values.")

    y_scaler = StandardScaler().fit(np.vstack(Y_list))
    x_scaler = StandardScaler().fit(np.vstack(X_list))
    z_scaler = StandardScaler().fit(np.vstack(Z_list)) if control_cols else None

    Y_list = [y_scaler.transform(y) for y in Y_list]
    X_list = [x_scaler.transform(x) for x in X_list]
    Z_list = [z_scaler.transform(z) for z in Z_list] if control_cols else Z_list

    return HMMData(Y_list, X_list, Z_list, ids, periods,
                   y_scaler, x_scaler, z_scaler)


# ---- Load & inspect ----
data = load_sequences(DATA_PATH)

seq_lens = [len(y) for y in data.Y]
print(f"Managers loaded : {len(data.Y)}")
print(f"Total observations: {sum(seq_lens)}")
print(f"Sequence lengths : min={min(seq_lens)}, median={int(np.median(seq_lens))}, max={max(seq_lens)}")
print(f"Emission dims (D) : {data.Y[0].shape[1]}  {emission_cols}")
print(f"Trans. covars (P) : {data.X[0].shape[1]}  {transition_cols}")
print(f"Controls (K)      : {data.Z[0].shape[1]}  {control_cols}")


---
<a id="2-hidden-markov-model-estimation"></a>
## 2. Hidden Markov Model Estimation

<a id="21-model-structure-and-inference-algorithms"></a>
### 2.1 Model Structure and Inference Algorithms


In [ ]:
# ============================================================
# Pre-stack all sequences to 3-D tensors for batched estimation
# ============================================================
import numpy as np

Y_stack = np.stack(data.Y)   # (N, T, D)
X_stack = np.stack(data.X)   # (N, T, P)
Z_stack = np.stack(data.Z)   # (N, T, K)

N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = N * T

print(f"Stacked arrays:")
print(f"  Y_stack: {Y_stack.shape}  (N={N}, T={T}, D={D})")
print(f"  X_stack: {X_stack.shape}  (N={N}, T={T}, P={P})")
print(f"  Z_stack: {Z_stack.shape}  (N={N}, T={T}, K={K})")
print(f"  n_obs_total = {n_obs_total}")

In [ ]:
# Collinearity check for both transition specifications
for spec_name, spec_cols in TRANSITION_SPECS.items():
    data_check = load_sequences(DATA_PATH, transition_cols_override=spec_cols)
    X_check = np.stack(data_check.X)
    X_flat = X_check.reshape(-1, X_check.shape[-1])
    corr = np.corrcoef(X_flat.T)
    df_corr = pd.DataFrame(corr, index=spec_cols, columns=spec_cols)

    def color_high(val):
        return "background-color: #ff4444; color: white" if abs(val) >= 0.80 and abs(val) < 1.0 else ""

    print()
    print(f"Transition collinearity: {spec_name}")
    display(df_corr.round(3).style.map(color_high))

    print("Pairs with |r| >= 0.70:")
    found = False
    for i in range(len(spec_cols)):
        for j in range(i + 1, len(spec_cols)):
            r = corr[i, j]
            if abs(r) >= 0.70:
                print(f"  {spec_cols[i]:40s}  {spec_cols[j]:40s}  r={r:.3f}")
                found = True
    if not found:
        print("  None - all pairs below 0.70")


In [ ]:
# ============================================================
# 4. Parameters + Forward–Backward  (VECTORIZED)
# ============================================================

@dataclass
class Params:
    logit_pi: np.ndarray   # (J,)
    alpha: np.ndarray      # (J, J)
    beta: np.ndarray       # (J, J, P)
    mu: np.ndarray         # (J, D)
    W: np.ndarray          # (J, D, K)
    log_sigma: np.ndarray  # (J, D)


def _precompute(p, Y, X, Z):
    """Shared emission + transition pre-computation."""
    T, D = Y.shape
    J = p.mu.shape[0]
    means = p.mu[None, :, :] + np.einsum('jdk,tk->tjd', p.W, Z)
    residuals = Y[:, None, :] - means
    sigma2 = np.exp(2 * p.log_sigma)
    log_norm = np.sum(np.log(2 * np.pi * sigma2), axis=1)
    logB = -0.5 * (log_norm[None, :] +
                   np.sum(residuals ** 2 / sigma2[None, :, :], axis=2))
    logits_all = (p.alpha[None, :, :]
                  + np.einsum('ijp,tp->tij', p.beta, X))
    logQ_all = log_softmax(logits_all, axis=2)
    return T, J, logB, logQ_all


def forward_only(p, Y, X, Z):
    """Forward pass only — returns log-likelihood (no posterior). ~2× faster."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    return float(logsumexp(log_alpha[-1]))


def forward_backward(p, Y, X, Z):
    """Full forward–backward returning (ll, log_gamma)."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    ll = logsumexp(log_alpha[-1])

    log_beta = np.zeros((T, J))
    for t in reversed(range(T - 1)):
        log_beta[t] = logsumexp(
            logQ_all[t + 1] + logB[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1)

    log_gamma = log_alpha + log_beta
    log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
    return ll, log_gamma

print("Section 4 — Params, forward_only() & forward_backward() defined.")

<a id="22-batched-maximum-likelihood-estimator"></a>
### 2.2 Batched Maximum Likelihood Estimator

Estimate non-homogeneous HMMs with Gaussian emissions and covariate-dependent transition probabilities using L-BFGS-B. The estimator supports subset screening, warm starts, exact continuation from checkpoints, and strict convergence checks for final model comparison.


In [ ]:
# ============================================================
# 5. Batched MLE Estimator  (fit_model_batched)
# ============================================================
# Accepts Y_stack, X_stack, Z_stack as EXPLICIT parameters so
# that Stack dimensions are always consistent with what was passed.
# Supports do_emission_only_warmstart for a two-phase init.
# ============================================================

import time
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp, log_softmax


def fit_model_batched(
    J: int,
    Y_stack: np.ndarray,
    X_stack: np.ndarray,
    Z_stack: np.ndarray,
    *,
    maxiter: int = 600,
    n_starts: int = 5,
    seed: int = 7,
    sigma_min: float = 0.05,
    sigma_max: float = 5.0,
    time_cap_min: int = 15,
    print_every: int = 50,
    l2: float = 1e-4,
    diag_bias: float = 2.0,
    maxfun: int = 200_000,
    ftol: float = 1e-8,
    gtol: float = 5e-6,
    warm_starts: list | None = None,
    use_subset: bool = False,
    subset_size: int = 80,
    do_emission_only_warmstart: bool = True,
    emission_only_maxiter: int = 120,
    emission_only_maxfun: int = 60_000,
    jitter_warm_starts: bool = True,
):
    """
    Batched NH-HMM fit with Gaussian emissions and covariate-dependent
    transitions via L-BFGS-B.

    Parameters
    ----------
    Y_stack : (N, T, D) emission data
    X_stack : (N, T, P) transition covariates
    Z_stack : (N, T, K) emission controls
    do_emission_only_warmstart : if True, first optimize emission params only
        (mu, W, log_sigma) with transitions fixed, then use as init.
    jitter_warm_starts : if True, perturb warm starts before fitting. Set
        False for exact continuation of an unfinished candidate.
    """
    # Derive dimensions from passed stacks
    N_full, T, D = Y_stack.shape
    P = X_stack.shape[2]
    K = Z_stack.shape[2]

    # Optional subset for speed
    if use_subset and N_full > subset_size:
        rng_sub = np.random.default_rng(seed)
        idx = rng_sub.choice(N_full, size=subset_size, replace=False)
        Y_use = Y_stack[idx]
        X_use = X_stack[idx]
        Z_use = Z_stack[idx]
        N_use = subset_size
    else:
        Y_use, X_use, Z_use = Y_stack, X_stack, Z_stack
        N_use = N_full

    # ── pack / unpack ──
    def pack(p):
        return np.concatenate([
            p.logit_pi.ravel(), p.alpha.ravel(), p.beta.ravel(),
            p.mu.ravel(), p.W.ravel(), p.log_sigma.ravel(),
        ])

    def unpack(theta):
        idx = 0
        def take(n):
            nonlocal idx; v = theta[idx:idx+n]; idx += n; return v
        return Params(
            logit_pi=take(J),
            alpha=take(J*J).reshape(J,J),
            beta=take(J*J*P).reshape(J,J,P),
            mu=take(J*D).reshape(J,D),
            W=take(J*D*K).reshape(J,D,K),
            log_sigma=take(J*D).reshape(J,D),
        )

    # ── bounds (log_sigma only) ──
    n_logit_pi = J
    n_alpha = J*J
    n_beta = J*J*P
    n_mu = J*D
    n_W = J*D*K
    n_log_sigma = J*D
    log_sigma_start = n_logit_pi + n_alpha + n_beta + n_mu + n_W
    log_sigma_end = log_sigma_start + n_log_sigma
    total_params = log_sigma_end

    LOW, HIGH = np.log(sigma_min), np.log(sigma_max)
    bounds = [(None, None)] * total_params
    for i in range(log_sigma_start, log_sigma_end):
        bounds[i] = (LOW, HIGH)

    # ── smart init ──
    Y_flat = Y_use.reshape(-1, D)
    y_mean = Y_flat.mean(axis=0)
    y_std = np.maximum(Y_flat.std(axis=0), 1e-3)

    def smart_init_params(rng):
        mu0 = y_mean[None,:] + rng.normal(0, 1.0, (J,D)) * y_std[None,:]
        log_sigma0 = np.log(np.clip(y_std, sigma_min, sigma_max))[None,:]
        log_sigma0 = np.repeat(log_sigma0, J, axis=0)
        log_sigma0 = np.clip(log_sigma0 + rng.normal(0, 0.12, (J,D)), LOW, HIGH)
        logit_pi0 = rng.normal(0, 0.2, J)
        alpha0 = rng.normal(0, 0.20, (J,J)) + np.eye(J) * diag_bias
        beta0 = rng.normal(0, 0.02, (J,J,P))
        W0 = rng.normal(0, 0.03, (J,D,K))
        return Params(logit_pi=logit_pi0, alpha=alpha0, beta=beta0,
                      mu=mu0, W=W0, log_sigma=log_sigma0)

    # ── neg-LL (batched forward algorithm) ──
    stop_flag = {"stop": False}

    def neg_ll(theta):
        if stop_flag["stop"]:
            return 1e50
        p = unpack(theta)
        log_pi = log_softmax(p.logit_pi, axis=0)
        means = p.mu[None,None,:,:] + np.einsum("jdk,ntk->ntjd", p.W, Z_use)
        resid = Y_use[:,:,None,:] - means
        sigma2 = np.maximum(np.exp(2.0 * p.log_sigma), 1e-6)
        log_norm = np.sum(np.log(2*np.pi * sigma2), axis=1)
        logB = -0.5 * (log_norm[None,None,:] +
                       np.sum(resid**2 / sigma2[None,None,:,:], axis=3))
        logQ = log_softmax(
            p.alpha[None,None,:,:] + np.einsum("ijp,ntp->ntij", p.beta, X_use),
            axis=3)
        la = log_pi[None,:] + logB[:,0,:]
        for t in range(1, T):
            la = logB[:,t,:] + logsumexp(la[:,:,None] + logQ[:,t,:,:], axis=1)
        ll = np.sum(logsumexp(la, axis=1))
        return -float(ll) if np.isfinite(ll) else 1e40

    def objective(theta):
        base = neg_ll(theta)
        if not np.isfinite(base) or l2 <= 0:
            return base if np.isfinite(base) else 1e40
        p = unpack(theta)
        pen = (np.sum(p.alpha**2) + np.sum(p.beta**2) +
               np.sum(p.W**2) + 0.10*np.sum(p.mu**2))
        return base + l2 * pen

    # ── emission-only warmstart objective ──
    def emission_only_objective(em_theta, fixed_logit_pi, fixed_alpha, fixed_beta):
        """Optimize only mu, W, log_sigma with transitions frozen."""
        idx = 0
        def take(n):
            nonlocal idx; v = em_theta[idx:idx+n]; idx += n; return v
        mu = take(J*D).reshape(J,D)
        W = take(J*D*K).reshape(J,D,K)
        log_sigma = take(J*D).reshape(J,D)
        p = Params(logit_pi=fixed_logit_pi, alpha=fixed_alpha,
                   beta=fixed_beta, mu=mu, W=W, log_sigma=log_sigma)
        theta_full = pack(p)
        return objective(theta_full)

    # ── multi-start optimization ──
    init_list = list(warm_starts) if warm_starts else []
    runs = []

    for s in range(n_starts):
        rng = np.random.default_rng(seed + s)
        stop_flag["stop"] = False

        # Initialize
        if s < len(init_list):
            p0 = init_list[s]
            # Ensure dimensions match
            if p0.W.shape != (J, D, K):
                print(f"  ⚠️  start {s+1}: warm start W shape {p0.W.shape} "
                      f"!= ({J},{D},{K}), using random init")
                p0 = smart_init_params(rng)
            elif jitter_warm_starts:
                p0 = Params(
                    logit_pi=p0.logit_pi + rng.normal(0, 0.03, J),
                    alpha=p0.alpha + rng.normal(0, 0.03, (J,J)),
                    beta=p0.beta + rng.normal(0, 0.008, (J,J,P)),
                    mu=p0.mu + rng.normal(0, 0.05, (J,D)),
                    W=p0.W + rng.normal(0, 0.01, (J,D,K)),
                    log_sigma=np.clip(p0.log_sigma + rng.normal(0, 0.02, (J,D)),
                                      LOW, HIGH),
                )
        else:
            p0 = smart_init_params(rng)

        # Phase 1 (optional): emission-only warmstart
        if do_emission_only_warmstart and s >= len(init_list):
            em_theta0 = np.concatenate([
                p0.mu.ravel(), p0.W.ravel(), p0.log_sigma.ravel()])
            em_bounds = ([(None,None)]*(J*D + J*D*K) +
                         [(LOW,HIGH)]*(J*D))
            em_res = minimize(
                emission_only_objective, em_theta0,
                args=(p0.logit_pi, p0.alpha, p0.beta),
                method="L-BFGS-B", bounds=em_bounds,
                options={"maxiter": emission_only_maxiter,
                         "maxfun": emission_only_maxfun})
            # unpack emission params back
            eidx = 0
            def etake(n):
                nonlocal eidx; v = em_res.x[eidx:eidx+n]; eidx += n; return v
            p0 = Params(logit_pi=p0.logit_pi, alpha=p0.alpha,
                        beta=p0.beta,
                        mu=etake(J*D).reshape(J,D),
                        W=etake(J*D*K).reshape(J,D,K),
                        log_sigma=etake(J*D).reshape(J,D))

        # Phase 2: full optimization
        theta0 = pack(p0)
        start_time = time.time()
        iter_counter = {"i": 0}

        def callback(_xk):
            iter_counter["i"] += 1
            if iter_counter["i"] % print_every == 0:
                elapsed_min = (time.time() - start_time) / 60
                print(f"    J={J} start {s+1}/{n_starts} "
                      f"iter={iter_counter['i']} elapsed={elapsed_min:.1f} min",
                      flush=True)
            if (time.time() - start_time) > time_cap_min * 60:
                stop_flag["stop"] = True

        res = minimize(objective, theta0, method="L-BFGS-B",
                       bounds=bounds, callback=callback,
                       options={"maxiter": maxiter, "maxfun": maxfun,
                                "ftol": ftol, "gtol": gtol})

        if stop_flag["stop"]:
            res.success = False
            res.message = f"Time cap reached ({time_cap_min} min)"

        true_negll = neg_ll(res.x)
        runs.append((unpack(res.x), res, true_negll))
        print(f"    done: J={J} start {s+1}/{n_starts} success={res.success} "
              f"nit={getattr(res,'nit',None)} true_negLL={true_negll:.2f} "
              f"msg={res.message}", flush=True)

    # ── pick best run ──
    converged = [(p,r,tnl) for (p,r,tnl) in runs if bool(r.success)]
    if converged:
        best_p, best_res, best_tnl = min(converged, key=lambda t: t[2])
        best_is_conv = True
    else:
        best_p, best_res, best_tnl = min(runs, key=lambda t: t[2])
        best_is_conv = False

    best_res.true_negll = best_tnl
    best_res.true_ll = -best_tnl
    best_res.k_params = len(best_res.x)
    return best_p, best_res, best_is_conv


print("fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.")

<a id="23-converged-v3-model-selection"></a>
### 2.3 Converged V3 Model Selection

Compare `J=2`, `J=3`, and `J=4` for both transition specifications. Each candidate is continued until SciPy reports optimizer success and the notebook's strict convergence flag is true. Final comparison uses LL, AIC, and BIC, with the selected model chosen as the simplest converged candidate within 10 BIC points of the best converged fit.


In [ ]:
# ============================================================
# V3 model comparison: single-period vs 3-period, J=2/3/4
# ============================================================

import os
import time
import pickle
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

try:
    display
except NameError:  # allows this cell to run from a plain Python runner
    display = print

J_CANDIDATES = [2, 3]
MODEL_SPECS = dict(TRANSITION_SPECS)

# All candidates are now run as continuation fits until SciPy reports
# convergence. The first phase builds a warm start on a subset; the second
# phase repeatedly resumes the full-data fit exactly from the previous point.
SCREEN_CONFIG_BY_J = {
    2: dict(maxiter=90, maxfun=80_000, diag_bias=2.4, subset_size=96, n_starts=3),
    3: dict(maxiter=120, maxfun=120_000, diag_bias=2.6, subset_size=96, n_starts=3),
    4: dict(maxiter=150, maxfun=160_000, diag_bias=2.8, subset_size=96, n_starts=3),
}

FULL_CONFIG_BY_J = {
    2: dict(maxiter=450, maxfun=350_000, diag_bias=2.4),
    3: dict(maxiter=600, maxfun=500_000, diag_bias=2.6),
    4: dict(maxiter=750, maxfun=650_000, diag_bias=2.8),
}

MAX_CONTINUATION_ROUNDS_BY_J = {2: 4, 3: 5, 4: 6}

COMMON_FIT_CONFIG = dict(
    sigma_min=0.1,
    sigma_max=3.5,
    print_every=75,
    time_cap_min=360,
    l2=0.01,
    ftol=1e-8,
    gtol=2e-6,
)

if Path.cwd().name == "HMM Estimation":
    _artifact_dir = Path("Artefacts")
else:
    _artifact_dir = Path("HMM Estimation") / "Artefacts"
_artifact_dir.mkdir(parents=True, exist_ok=True)
_data_signature = f"{DATA_PATH.resolve()}::{DATA_PATH.stat().st_size}::{DATA_PATH.stat().st_mtime_ns}"
_checkpoint_root = _artifact_dir / "Convergence_Checkpoints"
_checkpoint_dir = _checkpoint_root / hashlib.sha1(_data_signature.encode("utf-8")).hexdigest()[:16]
_checkpoint_dir.mkdir(parents=True, exist_ok=True)
_progress_path = _artifact_dir / "model_selection_v3_all_converged_progress.csv"
_final_comparison_path = _artifact_dir / "model_selection_v3_single_vs_3period.csv"
_existing_best_artifact_path = _artifact_dir / "best_model_artifacts_v3_2emissions.pkl"


def _candidate_checkpoint_path(spec_name, J):
    return _checkpoint_dir / f"candidate_v3_{spec_name}_J{J}.pkl"


def _model_metrics(res, n_obs):
    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = int(len(getattr(res, "x", [])))
    aic = 2.0 * k_params - 2.0 * ll_total
    bic = np.log(n_obs) * k_params - 2.0 * ll_total
    return ll_total, aic, bic, k_params


def _occupancy_metrics(p_hat, data_m):
    gammas = []
    for Y_i, X_i, Z_i in zip(data_m.Y, data_m.X, data_m.Z):
        _, log_g = forward_backward(p_hat, Y_i, X_i, Z_i)
        gammas.append(np.exp(log_g))
    gamma_all = np.concatenate(gammas, axis=0)
    return float(gamma_all.mean(axis=0).min()), float(gamma_all.max(axis=1).mean())


def _score_row(spec_name, transition_cols, J, p_hat, res, is_conv, data_m, runtime_min, round_idx, source, screen_converged=False):
    Y_m = np.stack(data_m.Y)
    n_obs_m = int(Y_m.shape[0] * Y_m.shape[1])
    ll_total, aic, bic, k_params = _model_metrics(res, n_obs_m)
    occupancy_min, certainty_mean = _occupancy_metrics(p_hat, data_m)
    return {
        "spec": spec_name,
        "J": int(J),
        "P": len(transition_cols),
        "transition_cols": ", ".join(transition_cols),
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "k_params": k_params,
        "n_obs": n_obs_m,
        "soft_converged": bool(is_conv),
        "scipy_success": bool(getattr(res, "success", False)),
        "strict_converged": bool(is_conv and getattr(res, "success", False)),
        "screen_converged": bool(screen_converged),
        "continuation_round": int(round_idx),
        "iterations": int(getattr(res, "nit", -1)) if getattr(res, "nit", None) is not None else np.nan,
        "occupancy_min": occupancy_min,
        "certainty_mean": certainty_mean,
        "runtime_min": runtime_min,
        "source": source,
        "message": str(getattr(res, "message", "")),
    }


def _write_progress(rows):
    if rows:
        pd.DataFrame(rows).sort_values(["spec", "J"]).to_csv(_progress_path, index=False)


def _save_checkpoint(spec_name, J, transition_cols, p_hat, res, is_conv, row, round_history):
    ckpt = {
        "spec": spec_name,
        "J": int(J),
        "transition_cols": list(transition_cols),
        "params": p_hat,
        "res": res,
        "is_conv": bool(is_conv),
        "row": row,
        "round_history": list(round_history),
    }
    with _candidate_checkpoint_path(spec_name, J).open("wb") as f:
        pickle.dump(ckpt, f)


def _load_checkpoint(spec_name, J, transition_cols):
    path = _candidate_checkpoint_path(spec_name, J)
    if not path.exists():
        return None
    with path.open("rb") as f:
        ckpt = pickle.load(f)
    if ckpt.get("spec") != spec_name or int(ckpt.get("J")) != int(J):
        return None
    if list(ckpt.get("transition_cols", [])) != list(transition_cols):
        return None
    return ckpt


def _load_existing_final_artifact(spec_name, J, transition_cols, data_m):
    if not _existing_best_artifact_path.exists():
        return None
    try:
        with _existing_best_artifact_path.open("rb") as f:
            art = pickle.load(f)
    except Exception as exc:
        print(f"Existing final artifact not reusable: {exc}", flush=True)
        return None

    if art.get("data_signature") != _data_signature:
        return None

    if art.get("best_spec") != spec_name or int(art.get("best_J", -1)) != int(J):
        return None
    if list(art.get("transition_cols", [])) != list(transition_cols):
        return None

    p_hat = art.get("best_model")
    res = art.get("best_res_final")
    is_conv = bool(art.get("best_is_conv_final") and getattr(res, "success", False))
    if p_hat is None or res is None or not is_conv:
        return None

    row = _score_row(
        spec_name, transition_cols, J, p_hat, res, is_conv, data_m,
        runtime_min=0.0, round_idx=0, source="existing_final_artifact",
        screen_converged=True,
    )
    return {
        "params": p_hat,
        "res": res,
        "is_conv": is_conv,
        "row": row,
        "round_history": [row],
    }


def fit_until_converged(spec_name, transition_cols, J, seed):
    data_m = load_sequences(DATA_PATH, transition_cols_override=transition_cols)
    Y_m = np.stack(data_m.Y)
    X_m = np.stack(data_m.X)
    Z_m = np.stack(data_m.Z)

    print()
    print(f"=== Candidate: spec={spec_name}, J={J}, P={len(transition_cols)} ===", flush=True)

    existing_final = _load_existing_final_artifact(spec_name, J, transition_cols, data_m)
    if existing_final is not None:
        print("  Reusing already converged final artifact for this candidate.", flush=True)
        row = existing_final["row"]
        _save_checkpoint(spec_name, J, transition_cols, existing_final["params"], existing_final["res"], True, row, existing_final["round_history"])
        return row, existing_final["params"], existing_final["res"], True, data_m

    ckpt = _load_checkpoint(spec_name, J, transition_cols)
    if ckpt is not None:
        p_current = ckpt["params"]
        res_current = ckpt["res"]
        round_history = list(ckpt.get("round_history", []))
        start_round = int(ckpt.get("row", {}).get("continuation_round", len(round_history))) + 1
        screen_converged = bool(ckpt.get("row", {}).get("screen_converged", False))
        print(f"  Resuming checkpoint at continuation round {start_round}.", flush=True)
        if bool(ckpt.get("is_conv")) and bool(getattr(res_current, "success", False)):
            return ckpt["row"], p_current, res_current, True, data_m
    else:
        print("  Screening warm starts on a manager subset.", flush=True)
        screen_cfg = dict(COMMON_FIT_CONFIG)
        screen_cfg.update({k: v for k, v in SCREEN_CONFIG_BY_J[J].items() if k != "n_starts"})
        t_screen = time.time()
        p_current, res_screen, screen_converged = fit_model_batched(
            J=J,
            Y_stack=Y_m,
            X_stack=X_m,
            Z_stack=Z_m,
            seed=seed,
            n_starts=SCREEN_CONFIG_BY_J[J]["n_starts"],
            use_subset=True,
            do_emission_only_warmstart=True,
            emission_only_maxiter=60,
            emission_only_maxfun=40_000,
            jitter_warm_starts=True,
            **screen_cfg,
        )
        print(
            f"  Screening complete: converged={screen_converged}, "
            f"runtime={(time.time() - t_screen) / 60.0:.1f} min",
            flush=True,
        )
        round_history = []
        start_round = 1

    full_cfg = dict(COMMON_FIT_CONFIG)
    full_cfg.update(FULL_CONFIG_BY_J[J])
    max_rounds = MAX_CONTINUATION_ROUNDS_BY_J[J]
    final_row = None
    final_res = None
    final_is_conv = False

    for round_idx in range(start_round, max_rounds + 1):
        print(
            f"  Full-data continuation round {round_idx}/{max_rounds} "
            f"(maxiter={full_cfg['maxiter']}, maxfun={full_cfg['maxfun']})",
            flush=True,
        )
        t0 = time.time()
        p_hat, res, is_conv = fit_model_batched(
            J=J,
            Y_stack=Y_m,
            X_stack=X_m,
            Z_stack=Z_m,
            seed=seed + 10_000 + round_idx,
            n_starts=1,
            warm_starts=[p_current],
            use_subset=False,
            do_emission_only_warmstart=False,
            emission_only_maxiter=0,
            jitter_warm_starts=False,
            **full_cfg,
        )
        runtime_min = (time.time() - t0) / 60.0
        row = _score_row(
            spec_name, transition_cols, J, p_hat, res, is_conv, data_m,
            runtime_min=runtime_min, round_idx=round_idx,
            source="full_data_continuation", screen_converged=screen_converged,
        )
        round_history.append(row)
        _save_checkpoint(spec_name, J, transition_cols, p_hat, res, is_conv, row, round_history)

        print(
            f"  Round {round_idx}: LL={row['LL']:.2f}, AIC={row['AIC']:.2f}, "
            f"BIC={row['BIC']:.2f}, converged={row['strict_converged']}, "
            f"runtime={runtime_min:.1f} min",
            flush=True,
        )

        final_row, final_res, final_is_conv = row, res, bool(is_conv and getattr(res, "success", False))
        p_current = p_hat
        if final_is_conv:
            break

    if final_row is None:
        raise RuntimeError(f"No full-data fit was produced for spec={spec_name}, J={J}")

    if not final_is_conv:
        print(
            f"  WARNING: spec={spec_name}, J={J} did not strictly converge "
            f"within {max_rounds} continuation rounds.",
            flush=True,
        )

    return final_row, p_current, final_res, final_is_conv, data_m


all_rows = []
fitted_models = {}
seed_base = 2026

for spec_idx, (spec_name, transition_cols_for_model) in enumerate(MODEL_SPECS.items()):
    for J in J_CANDIDATES:
        row, p_hat, res, is_conv, data_m = fit_until_converged(
            spec_name=spec_name,
            transition_cols=transition_cols_for_model,
            J=J,
            seed=seed_base + 100 * spec_idx + J,
        )
        all_rows.append(row)
        _write_progress(all_rows)
        fitted_models[(spec_name, J)] = {
            "params": p_hat,
            "res": res,
            "is_conv": bool(is_conv),
            "data": data_m,
            "transition_cols": list(transition_cols_for_model),
        }

results_ext = pd.DataFrame(all_rows).sort_values(["spec", "J"]).reset_index(drop=True)
results_ext.to_csv(_final_comparison_path, index=False)

print()
print("Model comparison: LL/AIC/BIC")
display_cols = [
    "spec", "J", "LL", "AIC", "BIC", "k_params", "n_obs",
    "strict_converged", "soft_converged", "scipy_success", "iterations",
    "continuation_round", "occupancy_min", "certainty_mean", "runtime_min", "source", "message",
]
display(results_ext[display_cols].round(4))

not_converged = results_ext[~results_ext["strict_converged"]].copy()
if not not_converged.empty:
    print()
    print("WARNING: Some candidates are still not strictly converged after continuation:")
    display(not_converged[["spec", "J", "LL", "BIC", "iterations", "message"]])

converged_results = results_ext[results_ext["strict_converged"]].copy()
if converged_results.empty:
    selection_pool = results_ext.copy()
    selection_note = "No strictly converged candidate; selected lowest BIC among all fits."
else:
    best_bic = float(converged_results["BIC"].min())
    selection_pool = converged_results[converged_results["BIC"] <= best_bic + 10.0].copy()
    selection_note = "Selected simplest strictly converged model within 10 BIC points of the best converged fit among theory-constrained J=2/3 candidates."

best_screen_row = selection_pool.sort_values(["J", "BIC"]).iloc[0]
best_spec = str(best_screen_row["spec"])
best_J_screen = int(best_screen_row["J"])
best_extension_choice = best_screen_row.to_dict()

best_entry = fitted_models[(best_spec, best_J_screen)]
best_model = best_entry["params"]
best_p_final = best_model
best_res_final = best_entry["res"]
best_is_conv_final = bool(best_entry["is_conv"])
data = best_entry["data"]
transition_cols_final = list(best_entry["transition_cols"])
Y_stack = np.stack(data.Y)
X_stack = np.stack(data.X)
Z_stack = np.stack(data.Z)
N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = int(N * T)
ll_total = float(best_screen_row["LL"])
aic = float(best_screen_row["AIC"])
bic = float(best_screen_row["BIC"])
k_params = int(best_screen_row["k_params"])
best_J = best_J_screen
BEST_J = best_J_screen

print()
print("Selected model:")
print(f"  spec={best_spec}")
print(f"  J={best_J_screen}")
print(f"  LL={ll_total:.2f}")
print(f"  AIC={aic:.2f}")
print(f"  BIC={bic:.2f}")
print(f"  converged={best_is_conv_final}")
print(f"  transition cols: {transition_cols_final}")
print("  rule:", selection_note)
print(f"Saved converged comparison to {_final_comparison_path.resolve()}")


---
<a id="3-final-model-diagnostics-and-artifact-export"></a>
## 3. Final Model Diagnostics and Artifact Export

The selected model is the full-data converged candidate from the V3 model-selection comparison. The final cells report model dimensions, diagnostics, and exported artifacts for downstream analysis.

<a id="31-selected-model-data-and-dimensions"></a>
### 3.1 Selected Model Data and Dimensions


In [ ]:
# ============================================================
# Selected model data and dimensions
# ============================================================

print("Selected model data ready.")
print(f"  selected spec: {best_spec}")
print(f"  selected J: {best_J}")
print(f"  transition cols ({P}): {transition_cols_final}")
print(f"  emission cols ({D}): {emission_cols}")
print(f"  control cols ({K}): {control_cols}")
print(f"  Y_stack={Y_stack.shape}, X_stack={X_stack.shape}, Z_stack={Z_stack.shape}")
print(f"  n_obs_total={n_obs_total}")


In [ ]:
# Final selected model size from full V3 comparison
print(f"Using best_J = {best_J} for the selected {best_spec} model.")


<a id="32-final-selected-fit-diagnostics"></a>
### 3.2 Final Selected Fit Diagnostics

Report the selected specification, fit statistics, convergence status, and optimizer message for the final model.


In [ ]:
# ============================================================
# Final selected fit diagnostics
# ============================================================

print()
print("=" * 60)
print(f"FINAL SELECTED MODEL: spec={best_spec}, J={best_J}")
print(f"LL:  {ll_total:.2f}")
print(f"AIC: {aic:.2f}")
print(f"BIC: {bic:.2f}")
print(f"Soft-converged: {bool(best_is_conv_final)}")
print(f"SciPy success:  {bool(getattr(best_res_final, 'success', False))}")
print(f"Message: {getattr(best_res_final, 'message', '')}")
print("=" * 60)


<a id="33-artifact-export"></a>
### 3.3 Artifact Export

Save the converged model comparison table and the selected best-model artifact used by downstream analysis notebooks.


In [ ]:
import pickle

# Save best model artifacts and the full J/spec comparison for later reuse.

model_artifacts = {
    "dataset_path": str(DATA_PATH.resolve()),
    "dataset_version": "v3",
    "data_signature": globals().get("_data_signature", None),
    "model_spec": "two_emission_v3_single_vs_3period",
    "best_spec": best_spec,
    "best_model": best_model,
    "best_J": int(best_J),
    "best_res_final": globals().get("best_res_final", None),
    "best_is_conv_final": globals().get("best_is_conv_final", None),
    "final_cfg": {
        "screen_config_by_J": globals().get("SCREEN_CONFIG_BY_J", None),
        "full_config_by_J": globals().get("FULL_CONFIG_BY_J", None),
        "common_fit_config": globals().get("COMMON_FIT_CONFIG", None),
        "selection_rule": globals().get("selection_note", None),
        "candidate_set": globals().get("J_CANDIDATES", None),
    },
    "model_comparison_results": globals().get("results_ext", None),
    "screening_results": globals().get("results_ext", None),
    "screening_choice": globals().get("best_extension_choice", None),
    "transition_specs": globals().get("TRANSITION_SPECS", None),
    "three_period_definition": globals().get("THREE_PERIOD_DEFINITION", None),
    "ll_total": globals().get("ll_total", None),
    "k_params": globals().get("k_params", None),
    "n_obs_total": globals().get("n_obs_total", None),
    "aic": globals().get("aic", None),
    "bic": globals().get("bic", None),
    "emission_cols": globals().get("emission_cols", None),
    "transition_cols": globals().get("transition_cols_final", globals().get("transition_cols", None)),
    "control_cols": globals().get("control_cols", None),
    "variable_labels": globals().get("VARIABLE_LABELS", None),
    "manager_heterogeneity_xi": 0.6447,
    "label_map": globals().get("label_map", None),
    "state_order_1idx": globals().get("state_order_1idx", None),
    "y_scaler": getattr(data, "y_scaler", None),
    "x_scaler": getattr(data, "x_scaler", None),
    "z_scaler": getattr(data, "z_scaler", None),
}

if Path.cwd().name == "HMM Estimation":
    artifact_dir = Path("Artefacts")
else:
    artifact_dir = Path("HMM Estimation") / "Artefacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

comparison_path = artifact_dir / "model_selection_v3_single_vs_3period.csv"
results_ext.to_csv(comparison_path, index=False)

out_path = artifact_dir / "best_model_artifacts_v3_2emissions.pkl"
with out_path.open("wb") as f:
    pickle.dump(model_artifacts, f)

print(f"Saved model comparison to {comparison_path.resolve()}")
print(f"Saved best model artifacts to {out_path.resolve()}")
